# Chapter 42: Image Classification with CNN
## Multimedia & Computer Vision Learning Hub — ITERA
**Author:** Aji Pamungkas Tri Nurcahyo
**Objective:** Membangun pipeline klasifikasi citra riil menggunakan dataset benchmark CIFAR-10, mengimplementasikan blok residual ResNet, mengevaluasi matriks kontingensi lengkap, serta membedah pencegahan data leakage dan overconfidence.

### 1. Landasan Teori & Formulasi Residual Learning
Dalam blok residual ResNet, pemetaan fungsi target didefinisikan sebagai:
$$\mathcal{H}(\mathbf{x}) = \mathcal{F}(\mathbf{x}, \{\mathbf{W}_i\}) + \mathbf{x}$$
Gradien propagasi balik mempertahankan komponen identitas bebas hambatan:
$$\frac{\partial \mathcal{E}}{\partial \mathbf{x}} = \frac{\partial \mathcal{E}}{\partial \mathcal{H}} \left( \frac{\partial \mathcal{F}}{\partial \mathbf{x}} + \mathbf{I} \right)$$

Metrik evaluasi multi-kelas per kategori:
$$\text{Precision}_c = \frac{TP_c}{TP_c + FP_c}, \quad \text{Recall}_c = \frac{TP_c}{TP_c + FN_c}, \quad F_{1, c} = \frac{2 \cdot P_c \cdot R_c}{P_c + R_c}$$

In [ ]:
# 2. Setup Lingkungan & Dual-Engine Check
import numpy as np
import sys

print(f"Python Version: {sys.version}")
print(f"NumPy Version: {np.__version__}")

HAS_TORCH = False
try:
    import torch
    import torch.nn as nn
    import torchvision
    import torchvision.transforms as transforms
    HAS_TORCH = True
    print(f"PyTorch: {torch.__version__}, Torchvision: {torchvision.__version__}")
except ImportError:
    print("INFO: PyTorch/torchvision tidak terdeteksi di lingkungan host saat ini.")
    print("Notebook beroperasi dalam Mode Analisis Algoritmik Deterministik (Offline Standalone).")

In [ ]:
# 3. Implementasi Blok Residual ResNet & Propagasi Nilai
def residual_block(x, weights_w1, weights_w2):
    z1 = np.dot(x, weights_w1)
    a1 = np.maximum(0, z1)
    f_x = np.dot(a1, weights_w2)
    h_x = f_x + x
    return np.maximum(0, h_x)

np.random.seed(42)
x_in = np.random.randn(4, 8)
w1 = np.random.randn(8, 8) * 0.1
w2 = np.random.randn(8, 8) * 0.1
out = residual_block(x_in, w1, w2)
print("Residual Block Output Shape:", out.shape)
print("Rata-rata Nilai Aktivasi Output:", np.round(np.mean(out), 4))
assert out.shape == (4, 8)

In [ ]:
# 4. Softmax Terkalibrasi & Categorical Cross-Entropy
def softmax(logits, temp=1.0):
    z = logits / temp
    e = np.exp(z - np.max(z, axis=-1, keepdims=True))
    return e / np.sum(e, axis=-1, keepdims=True)

logits = np.array([2.20, 1.10, -0.50])
p_std = softmax(logits, temp=1.0)
p_warm = softmax(logits, temp=2.0)

print("Standard Softmax (T=1.0):", np.round(p_std, 4))
print("Smoothed Softmax (T=2.0):", np.round(p_warm, 4))

# Loss for class 0 (True label)
loss = -np.log(p_std[0])
print(f"Cross-Entropy Loss (Class 0): {loss:.4f}")
assert np.isclose(loss, 0.3366, atol=1e-3)

In [ ]:
# 5. Evaluasi Matriks Kontingensi & Metrik Kuantitatif
cm = np.array([
    [35,  5,  2],
    [ 4, 28,  3],
    [ 1,  2, 20]
])

acc = np.trace(cm) / np.sum(cm)
precisions, recalls, f1s = [], [], []

for c in range(3):
    tp = cm[c, c]
    fp = np.sum(cm[:, c]) - tp
    fn = np.sum(cm[c, :]) - tp
    p = tp / (tp + fp)
    r = tp / (tp + fn)
    f = 2 * p * r / (p + r)
    precisions.append(p)
    recalls.append(r)
    f1s.append(f)

print(f"Akurasi Model: {acc*100:.2f}%")
print(f"Precision per Kelas: {[round(x, 4) for x in precisions]}")
print(f"Recall per Kelas:    {[round(x, 4) for x in recalls]}")
print(f"F1-Score per Kelas:  {[round(x, 4) for x in f1s]}")
print(f"Macro F1-Score:      {np.mean(f1s):.4f}")
assert np.isclose(acc, 0.83)
assert np.isclose(precisions[0], 0.8750)

In [ ]:
# 6. Real Deep Learning: CIFAR-10 Dataset Loader & ResNet Pipeline Specification
if HAS_TORCH:
    # Definisi pipeline augmentasi data bebas kebocoran (no data leakage)
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
    ])
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
    ])
    print("CIFAR-10 Augmentation & Normalization Pipeline Initialized:")
    print(transform_train)
    
    # Konstruksi ResNet-18 riil
    resnet18_model = torchvision.models.resnet18(weights=None, num_classes=10)
    print("\nTorchvision ResNet-18 Model Successfully Instantiated for 10 Classes:")
    print(f"Trainable Parameters: {sum(p.numel() for p in resnet18_model.parameters()):,}")
else:
    print("Spesifikasi Pipeline Pelatihan CIFAR-10 (Dokumentasi Protokol Riset):")
    print("1. Input: Citra RGB 32x32 piksel (50.000 data latih, 10.000 data uji).")
    print("2. Preprocessing: Random Crop (padding 4), Random Horizontal Flip.")
    print("3. Arsitektur Sasaran: ResNet-18 (11.17M parameter).")
    print("4. Optimasi: SGD (lr=0.1, momentum=0.9, weight_decay=5e-4, Cosine Annealing 200 epochs).")
    print("5. Jalankan pada Google Colab atau GPU workstation untuk mencapai Top-1 Acc ~95%.")

### 7. Analisis Hasil, Evaluasi & Rekomendasi
1. **Pencegahan Data Leakage**: Statistik normalisasi (mean dan std) wajib dihitung secara eksklusif dari training set dan tidak boleh menyertakan test set.
2. **Pentingnya Metrik Macro F1**: Akurasi tinggi tidak menjamin reliabilitas jika distribusi kelas timpang. Macro F1 memberikan pembobotan yang setara untuk seluruh kategori.
3. **Kalibrasi Suhu**: Mengontrol nilai temperatur $T$ pada Softmax sangat efektif dalam menghasilkan prediksi probabilitas yang realistis dan mencegah overconfidence pada sampel ambigu.